# Class Scheduling Algorithm
### Constraint-based Greedy Scheduler

## Cell 1 - Imports

In [ ]:
from dataclasses import dataclass, field
from typing import Optional
from itertools import product
import re
print('Imports ready')

Imports ready


## Cell 2 - Data Models

Example: `{'Mon': '00:00', 'Tue': '13:00', 'Wed': '14:00'}` means:
- Monday: available all day
- Tuesday: available from 13:00 onwards
- Wednesday: available from 14:00 onwards

In [ ]:
@dataclass
class Room:
    name: str
    capacity: int
    available: dict = field(default_factory=dict)

@dataclass
class Professor:
    name: str
    course: str
    students: int
    professor_type: str
    preferred_days: list[str]
    preferred_times: list[str]

@dataclass
class ScheduledClass:
    professor: Professor
    room: Room
    day: str
    time: str

@dataclass
class ScheduleResult:
    scheduled: list[ScheduledClass] = field(default_factory=list)
    unscheduled: list[tuple] = field(default_factory=list)

print('Data models defined')

Data models defined


## Cell 3 - Core Scheduler

`_find_room()` now checks two things before assigning a room:
1. The day must be in the room's `available` dict
2. The class time must be >= the room's earliest available time on that day


In [ ]:
class ClassScheduler:

    def __init__(self, rooms: list[Room], professors: list[Professor]):
        if not rooms:
            raise ValueError("At least one room is required.")
        if not professors:
            raise ValueError("At least one professor is required.")

        self.rooms      = rooms
        self.professors = professors

        # Auto-detect grid from actual professor data
        day_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
        all_days_set  = {d for p in professors for d in p.preferred_days}
        all_times_set = {t for p in professors for t in p.preferred_times}

        self.all_days  = [d for d in day_order if d in all_days_set]
        self.all_times = sorted(all_times_set)

        self._room_book = {
            r.name: {d: {t: None for t in self.all_times} for d in self.all_days}
            for r in self.rooms
        }
        self._prof_book = {
            p.name: {d: {t: False for t in self.all_times} for d in self.all_days}
            for p in self.professors
        }
        self._rooms_by_cap = sorted(self.rooms, key=lambda r: r.capacity)

    def run(self) -> ScheduleResult:
        result = ScheduleResult()
        # Warn about duplicate professor names
        seen, dups = set(), set()
        for p in self.professors:
            if p.name in seen:
                dups.add(p.name)
            seen.add(p.name)
        if dups:
            print(f"Warning: duplicate professor names detected: {dups}")
        for prof in self._sort_professors():
            self._schedule_professor(prof, result)
        return result

    def _sort_professors(self) -> list[Professor]:
        def key(p):
            return (
                len(p.preferred_days) * len(p.preferred_times),
                0 if p.professor_type == "external" else 1,
                -p.students,
            )
        return sorted(self.professors, key=key)

    def _schedule_professor(self, prof: Professor, result: ScheduleResult) -> None:
        last_reason = "No preferred day/time combination was available."

        for day, time in product(prof.preferred_days, prof.preferred_times):
            if day not in self.all_days or time not in self.all_times:
                last_reason = f"{day} {time} is not in the schedule grid."
                continue
            if self._prof_book[prof.name][day][time]:
                last_reason = f"Already assigned to another class on {day} at {time}."
                continue
            room = self._find_room(prof.students, day, time)
            if room is None:
                max_cap = max(r.capacity for r in self.rooms)
                if prof.students > max_cap:
                    last_reason = (
                        f"No room fits {prof.students} students. "
                        f"Largest room has {max_cap} seats."
                    )
                else:
                    last_reason = f"All suitable rooms occupied or unavailable on {day} at {time}."
                continue

            # Assign
            self._room_book[room.name][day][time] = prof.name
            self._prof_book[prof.name][day][time] = True
            result.scheduled.append(ScheduledClass(prof, room, day, time))
            return

        result.unscheduled.append((prof, last_reason))

    def _find_room(self, students: int, day: str, time: str) -> Optional[Room]:
        """
        Find the smallest room that:
          1. Has enough capacity
          2. Is available on this day
          3. The class time >= room earliest available time on that day
          4. Is not already booked at this slot
        """
        for room in self._rooms_by_cap:
            # Capacity check
            if room.capacity < students:
                continue
            # Availability check: day must be in room's available dict
            if day not in room.available:
                continue
            # Time restriction check: class time must be >= room's earliest time that day
            earliest = room.available[day]
            if time < earliest:
                continue
            # Booking check
            if self._room_book[room.name][day][time] is None:
                return room
        return None


print("ClassScheduler defined")

ClassScheduler defined


## Cell 4 - Output Functions

- `print_results()` - full scheduled table + unscheduled with reasons
- `print_timetable(scheduler)` - day x time grid, auto-sized to the data

In [ ]:
def print_results(result: ScheduleResult) -> None:
    print("=" * 80)
    print("SCHEDULE RESULTS")
    print("=" * 80)
    print(f"  Scheduled   : {len(result.scheduled)}")
    print(f"  Unscheduled : {len(result.unscheduled)}")
    print()

    if result.scheduled:
        print("-" * 80)
        print("SCHEDULED CLASSES")
        print("-" * 80)
        fmt = "{:<22} {:<22} {:<10} {:>8}  {:<5}  {:<7}  {}"
        print(fmt.format("Professor", "Course", "Type", "Students", "Day", "Time", "Room (cap)"))
        print("  " + "-" * 76)
        for s in result.scheduled:
            print(fmt.format(
                s.professor.name, s.professor.course, s.professor.professor_type,
                s.professor.students, s.day, s.time,
                f"{s.room.name} ({s.room.capacity})",
            ))

    if result.unscheduled:
        print()
        print("-" * 80)
        print("UNSCHEDULED PROFESSORS")
        print("-" * 80)
        for prof, reason in result.unscheduled:
            print(f"  {prof.name:<22} {prof.course:<22} {prof.professor_type:<10} students: {prof.students}")
            print(f"  Reason         : {reason}")
            print(f"  Preferred days : {prof.preferred_days}")
            print(f"  Preferred times: {prof.preferred_times}")
            print()

    print("=" * 80)


def print_timetable(result: ScheduleResult, scheduler: ClassScheduler) -> None:
    days  = scheduler.all_days
    times = scheduler.all_times

    grid = {(d, t): [] for d in days for t in times}
    for s in result.scheduled:
        if (s.day, s.time) in grid:
            grid[(s.day, s.time)].append(f"{s.professor.name}/{s.room.name}")

    col_w   = 30
    total_w = 7 + col_w * len(days)
    print("\n" + "=" * total_w)
    print("TIMETABLE")
    print("=" * total_w)
    print(f"{'Time':<7}" + "".join(f"{d:<{col_w}}" for d in days))
    print("-" * total_w)

    for t in times:
        entries = [", ".join(grid[(d, t)]) or "-" for d in days]
        max_lines = max(1, max((len(e) // (col_w - 2)) + 1 for e in entries))
        for line_i in range(max_lines):
            row = f"{t if line_i == 0 else '':<7}"
            for e in entries:
                chunk = e[line_i * (col_w - 2):(line_i + 1) * (col_w - 2)]
                row += f"{chunk:<{col_w}}"
            print(row)
        print()


print("Output functions defined")

Output functions defined


## Cell 5 - Load Data from Excel File

Upload `.xlsx` file using the file picker.

### Sheet 1 - Professors

### Sheet 2 - Rooms

In [ ]:
from google.colab import files
import io, re
import pandas as pd

print("Select your Excel file (.xlsx) ...")
uploaded = files.upload()

EXCEL_FILE = next(iter(uploaded))
print(f"Uploaded: {EXCEL_FILE}")

VALID_DAYS = {"Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"}

# Also accept common variants like "Tues", "Thurs" and normalize them
DAY_ALIASES = {
    "Tues": "Tue", "Thurs": "Thu", "Thur": "Thu",
}

def normalize_day(raw: str) -> str:
    raw = raw.strip()
    return DAY_ALIASES.get(raw, raw)

def normalize_time(raw: str) -> str:
    raw = str(raw).strip()
    parts = raw.split(":")
    if len(parts) >= 2:
        try:
            return f"{int(parts[0]):02d}:{parts[1].zfill(2)}"
        except ValueError:
            pass
    return raw

def parse_available(raw) -> dict:
    if not raw or str(raw).strip().lower() in ("nan", ""):
        # No restriction: available all days all day
        return {d: "00:00" for d in VALID_DAYS}

    result = {}
    tokens = re.split(r",(?![^(]*\))", str(raw))

    for token in tokens:
        token = token.strip()
        if not token:
            continue

        # Check for time restriction: "Day (from HH:MM)"
        match = re.match(r"([A-Za-z]+)\s*\(from\s*(\d{1,2}:\d{2})\)", token, re.IGNORECASE)
        if match:
            day  = normalize_day(match.group(1).strip().capitalize())
            time = normalize_time(match.group(2))
        else:
            day  = normalize_day(token.strip().capitalize())
            time = "00:00"

        if day not in VALID_DAYS:
            raise ValueError(f"Unknown day in available column: {token!r}. Got day={day!r}")

        result[day] = time

    return result

def parse_list(raw, label: str, row: int, valid_set=None) -> list:
    items = [v.strip() for v in str(raw).split(",") if v.strip()]
    if not items:
        raise ValueError(f"Row {row} - {label!r} is empty.")
    if valid_set:
        bad = [v for v in items if v not in valid_set]
        if bad:
            raise ValueError(f"Row {row} - invalid {label}: {bad}. Allowed: {sorted(valid_set)}")
    return items

def load_excel(file_bytes) -> tuple:
    xls = pd.ExcelFile(file_bytes)

    missing = {"Professors", "Rooms"} - set(xls.sheet_names)
    if missing:
        raise ValueError(f"Missing sheets: {missing}. File has: {xls.sheet_names}")

    # ---- Rooms ----
    df_rooms = pd.read_excel(xls, sheet_name="Rooms", skiprows=[1])
    df_rooms.columns = df_rooms.columns.str.strip().str.lower()
    df_rooms = df_rooms.dropna(how="all").reset_index(drop=True)

    miss = {"room_name", "capacity"} - set(df_rooms.columns)
    if miss:
        raise ValueError(f"Rooms sheet missing columns: {miss}")

    rooms = []
    for i, row in df_rooms.iterrows():
        er = i + 3
        name = str(row["room_name"]).strip()
        if not name or name.lower() == "nan":
            raise ValueError(f"Rooms row {er}: room_name is empty.")
        try:
            cap = int(float(str(row["capacity"])))
        except (ValueError, TypeError):
            raise ValueError(f"Rooms row {er}: capacity must be a whole number, got {row['capacity']!r}.")
        if cap <= 0:
            raise ValueError(f"Rooms row {er}: capacity must be > 0.")

        raw_avail = row.get("available", None)
        try:
            available = parse_available(raw_avail)
        except ValueError as e:
            raise ValueError(f"Rooms row {er} ({name}): {e}")

        rooms.append(Room(name=name, capacity=cap, available=available))

    # ---- Professors ----
    df_profs = pd.read_excel(xls, sheet_name="Professors", skiprows=[1])
    df_profs.columns = df_profs.columns.str.strip().str.lower()
    df_profs = df_profs.dropna(how="all").reset_index(drop=True)

    required = {"professor_name", "course_name", "num_students", "professor_type",
                "preferred_days", "preferred_times"}
    miss = required - set(df_profs.columns)
    if miss:
        raise ValueError(f"Professors sheet missing columns: {miss}")

    professors = []
    for i, row in df_profs.iterrows():
        er = i + 3
        name   = str(row["professor_name"]).strip()
        course = str(row["course_name"]).strip()
        ptype  = str(row["professor_type"]).strip().lower()

        if not name or name.lower() == "nan":
            raise ValueError(f"Professors row {er}: professor_name is empty.")
        if ptype not in {"external", "internal"}:
            raise ValueError(f"Professors row {er}: professor_type must be 'external' or 'internal', got {ptype!r}.")
        try:
            students = int(float(str(row["num_students"])))
        except (ValueError, TypeError):
            raise ValueError(f"Professors row {er}: num_students must be integer, got {row['num_students']!r}.")
        if students <= 0:
            raise ValueError(f"Professors row {er}: num_students must be > 0.")

        days      = parse_list(row["preferred_days"],  "preferred_days",  er, VALID_DAYS)
        raw_times = parse_list(row["preferred_times"], "preferred_times", er)
        times     = [normalize_time(t) for t in raw_times]

        professors.append(Professor(
            name=name, course=course, students=students,
            professor_type=ptype, preferred_days=days, preferred_times=times,
        ))

    return rooms, professors

try:
    file_data = io.BytesIO(uploaded[EXCEL_FILE])
    rooms, professors = load_excel(file_data)
    print(f"\n{len(rooms)} rooms and {len(professors)} professors loaded")

    print("\nRooms:")
    for r in rooms:
        avail_str = ", ".join(
            f"{d}(from {t})" if t != "00:00" else d
            for d, t in sorted(r.available.items())
        )
        print(f"  {r.name:<20} cap: {r.capacity:<5} available: {avail_str}")

    print("\nProfessors (first 10):")
    for p in professors[:10]:
        print(f"  {p.name:<22} {p.course:<22} {p.professor_type:<10} "
              f"students: {p.students:<4}  "
              f"days: {','.join(p.preferred_days):<18}  "
              f"times: {','.join(p.preferred_times)}")
    if len(professors) > 10:
        print(f"  ... and {len(professors) - 10} more")

except ValueError as e:
    print(f"\nError in Excel file:\n  {e}")
    print("Fix the issue and re-upload.")
except Exception as e:
    print(f"\nUnexpected error: {e}")

Select your Excel file (.xlsx) ...


Saving schedule_input_template.xlsx to schedule_input_template.xlsx
Uploaded: schedule_input_template.xlsx

10 rooms and 69 professors loaded

Rooms:
  Room 03              cap: 35    available: Mon
  Room 06              cap: 50    available: Mon, Tue, Wed
  Room 07              cap: 36    available: Fri, Mon, Sat, Thu, Tue, Wed
  Room 08              cap: 36    available: Fri, Mon, Sat, Thu, Tue, Wed
  Room 09              cap: 23    available: Fri, Mon, Sat, Thu, Tue, Wed
  Room 10              cap: 34    available: Fri, Mon, Sat, Thu, Tue, Wed
  Room 11              cap: 30    available: Fri
  Room 12              cap: 36    available: Mon, Thu
  Auditorium           cap: 60    available: Wed
  MacPool              cap: 25    available: Mon, Thu(from 13:00), Tue(from 13:00), Wed(from 14:00)

Professors (first 10):
  Ahmad                  Calculus               internal   students: 22    days: Mon                 times: 09:00
  Alexandru              Linear Algebra         internal

## Cell 6 - Run the Scheduler

Re-run this cell every time you upload new data.

In [ ]:
scheduler = ClassScheduler(rooms, professors)
result    = scheduler.run()

print('Scheduling complete!')
print(f'  Scheduled   : {len(result.scheduled)}')
print(f'  Unscheduled : {len(result.unscheduled)}')
print(f'\nAuto-detected grid:')
print(f'  Days  : {scheduler.all_days}')
print(f'  Times : {scheduler.all_times}')

Scheduling complete!
  Scheduled   : 54
  Unscheduled : 15

Auto-detected grid:
  Days  : ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
  Times : ['09:00', '12:00', '13:00', '15:00', '16:00']


## Cell 7 - View Detailed Results

In [ ]:
print_results(result)

SCHEDULE RESULTS
  Scheduled   : 54
  Unscheduled : 15

--------------------------------------------------------------------------------
SCHEDULED CLASSES
--------------------------------------------------------------------------------
Professor              Course                 Type       Students  Day    Time     Room (cap)
  ----------------------------------------------------------------------------
Samira                 Economics              external         55  Wed    12:00    Auditorium (60)
Alexander              Linear Algebra         external         34  Wed    09:00    Room 10 (34)
Lina                   Linear Algebra         external         34  Thu    12:00    Room 10 (34)
Anil                   Linear Algebra         external         34  Fri    15:00    Room 10 (34)
Nikos                  English                external         33  Sat    16:00    Room 10 (34)
Patricia               Computer Science       external         33  Thu    15:00    Room 10 (34)
Ahmed Khan  

## Cell 8 - Timetable, Room Availability & Schedule Verification

- **8a.** Prepares the data (grid, free slots) and runs the 6 self-verification checks
- **8b.** Section 1 - **Colour-coded timetable** of all scheduled classes
- **8c.** Section 2 - **Unscheduled professors** with the reason they could not be placed
- **8d.** Section 3 - **Free room slots** so unscheduled professors can be manually considered
- **8e.** Section 4 - **Schedule verification** result (violations, if any)

### 8a. Prepare timetable data & run self-verification checks

In [ ]:
from IPython.display import display, HTML

days  = scheduler.all_days
times = scheduler.all_times

# Build grid
grid = {(d, t): [] for d in days for t in times}
for s in result.scheduled:
    if (s.day, s.time) in grid:
        grid[(s.day, s.time)].append(s)
day_counts = {d: sum(len(grid[(d, t)]) for t in times) for d in days}

# Free room slots: open AND room available AND not booked
free_slots = {}
for room in scheduler.rooms:
    slots = []
    for d in days:
        if d not in room.available:
            continue
        earliest = room.available[d]
        for t in times:
            if t < earliest:
                continue
            if scheduler._room_book[room.name][d][t] is None:
                slots.append((d, t))
    if slots:
        free_slots[room.name] = (room.capacity, slots)

# Self-verification checks
violations = []

# Check 1: No room double-booking
room_time_map = {}
for s in result.scheduled:
    key = (s.room.name, s.day, s.time)
    if key in room_time_map:
        violations.append(
            f'DOUBLE-BOOKING: {s.professor.name} and {room_time_map[key]} '
            f'both in {s.room.name} on {s.day} at {s.time}.'
        )
    else:
        room_time_map[key] = s.professor.name

# Check 2: No professor teaching two classes at once
prof_time_map = {}
for s in result.scheduled:
    key = (s.professor.name, s.day, s.time)
    if key in prof_time_map:
        violations.append(
            f'PROFESSOR CONFLICT: {s.professor.name} assigned twice '
            f'on {s.day} at {s.time} ({prof_time_map[key]} and {s.room.name}).'
        )
    else:
        prof_time_map[key] = s.room.name

# Check 3: Room capacity not exceeded
for s in result.scheduled:
    if s.professor.students > s.room.capacity:
        violations.append(
            f'CAPACITY EXCEEDED: {s.professor.name} has {s.professor.students} students '
            f'but {s.room.name} fits only {s.room.capacity}.'
        )

# Check 4: Assignment on a preferred day
for s in result.scheduled:
    if s.day not in s.professor.preferred_days:
        violations.append(
            f'WRONG DAY: {s.professor.name} placed on {s.day} '
            f'but preferred {s.professor.preferred_days}.'
        )

# Check 5: Assignment at a preferred time
for s in result.scheduled:
    if s.time not in s.professor.preferred_times:
        violations.append(
            f'WRONG TIME: {s.professor.name} placed at {s.time} '
            f'but preferred {s.professor.preferred_times}.'
        )

# Check 6: Room available on assigned day and time
for s in result.scheduled:
    if s.day not in s.room.available:
        violations.append(
            f'ROOM UNAVAILABLE: {s.room.name} is not open on {s.day} '
            f'(assigned to {s.professor.name}).'
        )
    elif s.time < s.room.available[s.day]:
        violations.append(
            f'ROOM TIME RESTRICTION: {s.room.name} opens at {s.room.available[s.day]} '
            f'on {s.day} but {s.professor.name} placed at {s.time}.'
        )


print(f'Data ready: {len(result.scheduled)} scheduled classes, '
      f'{len(free_slots)} room(s) with free slots, {len(violations)} violation(s) found.')


Data ready: 54 scheduled classes, 10 room(s) with free slots, 0 violation(s) found.


### 8b. Section 1 — Colour-coded timetable

In [ ]:
css = '''
<style>
  .tt-wrap{font-family:'Segoe UI',Arial,sans-serif;margin:12px 0}
  .tt-header{background:#1e293b;color:#f8fafc;padding:14px 20px;border-radius:10px 10px 0 0;display:flex;align-items:baseline;gap:14px}
  .tt-header h2{margin:0;font-size:17px;font-weight:600}
  .tt-header span{font-size:12px;color:#94a3b8}
  .tt-legend{background:#f8fafc;border:1px solid #e2e8f0;border-top:none;padding:8px 20px;display:flex;gap:20px;flex-wrap:wrap;align-items:center}
  .tt-legend-item{display:flex;align-items:center;gap:6px;font-size:12px;color:#475569}
  .tt-dot{width:10px;height:10px;border-radius:3px;flex-shrink:0}
  .tt-dot-ext{background:#2563eb}.tt-dot-int{background:#16a34a}.tt-dot-empty{background:#e2e8f0}
  .tt-table{width:100%;border-collapse:collapse;border:1px solid #e2e8f0;border-top:none;overflow:hidden}
  .tt-table thead th{background:#f1f5f9;color:#334155;font-size:12px;font-weight:600;text-transform:uppercase;letter-spacing:.06em;padding:10px 14px;border-bottom:2px solid #e2e8f0;border-right:1px solid #e2e8f0;text-align:center;white-space:nowrap}
  .tt-table thead th.time-col{text-align:left;width:72px}
  .day-count{display:block;font-size:10px;font-weight:400;color:#94a3b8;text-transform:none;letter-spacing:0;margin-top:2px}
  .tt-table tbody tr:nth-child(even) td{background:#f8fafc}
  .tt-table tbody tr:nth-child(odd) td{background:#fff}
  .tt-table tbody td{padding:6px 8px;border-right:1px solid #e2e8f0;border-bottom:1px solid #f1f5f9;vertical-align:top;min-width:120px}
  .tt-table tbody td.time-cell{font-size:11px;font-weight:700;color:#64748b;background:#f1f5f9!important;border-right:2px solid #e2e8f0;white-space:nowrap;text-align:center;min-width:60px}
  .tt-empty{color:#cbd5e1;font-size:16px;text-align:center;display:block;padding:6px 0}
  .tt-card{border-radius:6px;padding:6px 8px;margin-bottom:4px;font-size:11px;line-height:1.4;border-left:3px solid transparent}
  .tt-card:last-child{margin-bottom:0}
  .tt-card-ext{background:#eff6ff;border-left-color:#2563eb;color:#1e3a5f}
  .tt-card-int{background:#f0fdf4;border-left-color:#16a34a;color:#14532d}
  .card-name{font-weight:700;font-size:11px}
  .card-course{font-size:10px;color:#64748b;margin-top:1px}
  .card-meta{display:flex;gap:6px;margin-top:4px;flex-wrap:wrap}
  .tt-badge{font-size:9px;font-weight:600;padding:1px 5px;border-radius:4px;background:rgba(0,0,0,.07);white-space:nowrap}
</style>
'''

display(HTML(css))

# Section 1: Colour-coded timetable
total = len(result.scheduled)
html  = '<div class="tt-wrap">'
html += (f'<div class="tt-header"><h2>Timetable</h2>'
         f'<span>{total} classes scheduled across {len(days)} day{"s" if len(days)!=1 else ""}</span></div>')
html += ('<div class="tt-legend">'
         '<div class="tt-legend-item"><div class="tt-dot tt-dot-ext"></div>External professor</div>'
         '<div class="tt-legend-item"><div class="tt-dot tt-dot-int"></div>Internal professor</div>'
         '<div class="tt-legend-item"><div class="tt-dot tt-dot-empty"></div>No class</div>'
         '</div>')
html += '<table class="tt-table"><thead><tr><th class="time-col">Time</th>'
for d in days:
    cnt = day_counts[d]
    html += f'<th>{d}<span class="day-count">{cnt} class{"es" if cnt!=1 else ""}</span></th>'
html += '</tr></thead><tbody>'
for t in times:
    html += f'<tr><td class="time-cell">{t}</td>'
    for d in days:
        entries = grid[(d, t)]
        if not entries:
            html += '<td><span class="tt-empty">-</span></td>'
        else:
            html += '<td>'
            for s in entries:
                cls = 'tt-card-ext' if s.professor.professor_type == 'external' else 'tt-card-int'
                html += (f'<div class="tt-card {cls}">'
                         f'<div class="card-name">{s.professor.name}</div>'
                         f'<div class="card-course">{s.professor.course}</div>'
                         f'<div class="card-meta">'
                         f'<span class="tt-badge">&#127963; {s.room.name}</span>'
                         f'<span class="tt-badge">&#128101; {s.professor.students}</span>'
                         f'</div></div>')
            html += '</td>'
    html += '</tr>'
html += '</tbody></table>'
html += '</div>'
display(HTML(html))


Time,Mon13 classes,Tue11 classes,Wed12 classes,Thu8 classes,Fri4 classes,Sat6 classes
09:00,DanielHistory🏛 Room 06👥 48AhmadCalculus🏛 Room 09👥 22RezaCalculus🏛 Room 10👥 30Andreas KellerEconomics🏛 MacPool👥 24FaridStatistics🏛 Room 03👥 15HamidChemistry🏛 Room 07👥 12,OmarPhilosophy🏛 Room 06👥 44CarstenStatistics🏛 Room 10👥 33,AlexanderLinear Algebra🏛 Room 10👥 34Ahmed KhanPhilosophy🏛 Room 07👥 28MinhEnglish🏛 Room 08👥 24Hossein RahimiComputer Science🏛 Room 09👥 18Stefan MüllerComputer Science🏛 Room 06👥 12RebeccaChemistry🏛 Auditorium👥 12,KavehPhilosophy🏛 Room 09👥 19,VarunCalculus🏛 Room 11👥 30,MazharCalculus🏛 Room 09👥 22ArashEconomics🏛 Room 10👥 28
12:00,Mohammad MadhaviLinear Algebra🏛 Room 06👥 38RajeshEnglish🏛 Room 09👥 22SaraEnglish🏛 Room 10👥 33,AlexandruLinear Algebra🏛 Room 06👥 38JonathanHistory🏛 Room 09👥 22,SamiraEconomics🏛 Auditorium👥 55LeilaBiology🏛 Room 09👥 18,LinaLinear Algebra🏛 Room 10👥 34BaoCalculus🏛 Room 09👥 18MaltePhilosophy🏛 Room 07👥 30,YasminChemistry🏛 Room 09👥 20,KarimComputer Science🏛 Room 09👥 20
13:00,BilalCalculus🏛 Room 10👥 30Charlotte GreenHistory🏛 Room 09👥 18KelvinLinear Algebra🏛 Room 06👥 47,Mohammad RezaeiCalculus🏛 Room 06👥 39,AymanChemistry🏛 Auditorium👥 55,SabineCalculus🏛 MacPool👥 24,ElenaEnglish🏛 Room 10👥 34,-
15:00,ViktoriaChemistry🏛 Room 06👥 38,-,RezaChemistry🏛 Auditorium👥 55SamiBiology🏛 Room 06👥 50,PatriciaComputer Science🏛 Room 10👥 33FarhadEnglish🏛 Room 09👥 20LukasPhilosophy🏛 MacPool👥 19,AnilLinear Algebra🏛 Room 10👥 34,JamesBiology🏛 Room 10👥 24
16:00,-,VikramPhilosophy🏛 Room 10👥 28LaurentPhilosophy🏛 Room 09👥 18Michael BrownComputer Science🏛 MacPool👥 10AndreasStatistics🏛 Room 07👥 33MatthiasComputer Science🏛 Room 08👥 30GiselaEnglish🏛 Room 06👥 20,VahabCalculus🏛 Room 09👥 20,-,-,NikosEnglish🏛 Room 10👥 33LauraComputer Science🏛 Room 09👥 19


### 8c. Section 2 — Unscheduled professors

In [ ]:
css = '''
<style>
  .tt-unsched{margin-top:16px; background: #ffffff; border:1px solid #fecaca; border-radius:10px; overflow:hidden}
  .tt-unsched-header{background:#fef2f2; padding:10px 16px; font-size:13px; font-weight:600; color:#991b1b; border-bottom:1px solid #fecaca}
  .tt-unsched-row{display:flex;gap:12px;padding:8px 16px;border-bottom:1px solid #fef2f2;font-size:12px;align-items:flex-start;flex-wrap:wrap}
  .tt-unsched-row:last-child{border-bottom:none}
  .tt-unsched-name{font-weight:600;color:#091a36;min-width:120px}
  .tt-unsched-course{color:#1867d9;min-width:200px}
  .tt-unsched-reason{color: #e00909;font-style:italic;flex:1;}
</style>
'''

display(HTML(css))

# Section 2: Unscheduled professors
html = '<div class="tt-wrap">'
if result.unscheduled:
    html += (f'<div class="tt-unsched">'
             f'<div class="tt-unsched-header">&#9888; {len(result.unscheduled)} '
             f'Unscheduled Professor{"s" if len(result.unscheduled)!=1 else ""}</div>')
    for prof, reason in result.unscheduled:
        html += (f'<div class="tt-unsched-row">'
                 f'<span class="tt-unsched-name">{prof.name}</span>'
                 f'<span class="tt-unsched-course">{prof.course} &bull; {prof.students} students</span>'
                 f'<span class="tt-unsched-reason">{reason}</span>'
                 f'</div>')
    html += '</div>'
else:
    html += '<div class="tt-verify-body" style="color:#15803d">All professors were successfully scheduled.</div>'
html += '</div>'
display(HTML(html))


### 8d. Section 3 — Free room slots

In [20]:
css = '''
<style>
  .tt-free{margin-top:16px;border:1px solid #d1fae5;border-radius:10px;overflow:hidden}
  .tt-free-header{background:#ecfdf5;padding:10px 16px;font-size:13px;font-weight:600;color:#065f46;border-bottom:1px solid #d1fae5}
  .tt-free-sub{background:#ecfdf5;padding:4px 16px 8px;font-size:11px;color:#6b7280;border-bottom:1px solid #d1fae5}
  .tt-free-table{width:100%;border-collapse:collapse;font-size:12px}
  .tt-free-table th{background:#f0fdf4;padding:8px 12px;text-align:left;font-size:11px;font-weight:600;color:#065f46;text-transform:uppercase;letter-spacing:.05em;border-bottom:1px solid #d1fae5}
  .tt-free-table td{padding:8px 12px;border-bottom:1px solid #f0fdf4;vertical-align:top;color:#387ae4}
  .tt-free-table tr:last-child td{border-bottom:none}
  .tt-free-table tr:nth-child(even) td{background:#f9fffe}
  .tt-slot-pill{display:inline-block;background:#d1fae5;color:#065f46;font-size:10px;font-weight:600;padding:2px 7px;border-radius:10px;margin:2px 3px 2px 0;white-space:nowrap}
  .tt-cap-badge{background:#e0f2fe;color:#0369a1;font-size:10px;font-weight:600;padding:2px 7px;border-radius:10px}
  .tt-verify-body{padding:10px 16px;background:#fff;font-size:12px}
</style>
'''

display(HTML(css))

# Section 3: Free room slots
html = '<div class="tt-wrap">'
html += ('<div class="tt-free">'
         '<div class="tt-free-header">&#128197; Available Room Slots</div>'
         '<div class="tt-free-sub">Unbooked slots — can be offered to unscheduled professors</div>')
if free_slots:
    html += ('<table class="tt-free-table"><thead><tr>'
             '<th>Room</th><th>Capacity</th><th>Free Slots</th>'
             '</tr></thead><tbody>')
    for room_name, (cap, slots) in sorted(free_slots.items()):
        by_day = {}
        for d, t in slots:
            by_day.setdefault(d, []).append(t)
        pills = ''
        for d in days:
            if d not in by_day:
                continue
            for t in sorted(by_day[d]):
                pills += f'<span class="tt-slot-pill">{d} {t}</span>'
        html += (f'<tr>'
                 f'<td><strong>{room_name}</strong></td>'
                 f'<td><span class="tt-cap-badge">&#128101; {cap}</span></td>'
                 f'<td>{pills}</td>'
                 f'</tr>')
    html += '</tbody></table>'
else:
    html += '<div class="tt-verify-body" style="color:#15803d">All room slots are fully booked.</div>'
html += '</div>'

display(HTML(html))


Room,Capacity,Free Slots
Auditorium,👥 60,Wed 16:00
MacPool,👥 25,Mon 12:00Mon 13:00Mon 15:00Mon 16:00Tue 13:00Tue 15:00Wed 15:00Wed 16:00Thu 16:00
Room 03,👥 35,Mon 12:00Mon 13:00Mon 15:00Mon 16:00
Room 06,👥 50,Mon 16:00Tue 15:00Wed 12:00Wed 13:00Wed 16:00
Room 07,👥 36,Mon 12:00Mon 13:00Mon 15:00Mon 16:00Tue 09:00Tue 12:00Tue 13:00Tue 15:00Wed 12:00Wed 13:00Wed 15:00Wed 16:00Thu 09:00Thu 13:00Thu 15:00Thu 16:00Fri 09:00Fri 12:00Fri 13:00Fri 15:00Fri 16:00Sat 09:00Sat 12:00Sat 13:00Sat 15:00Sat 16:00
Room 08,👥 36,Mon 09:00Mon 12:00Mon 13:00Mon 15:00Mon 16:00Tue 09:00Tue 12:00Tue 13:00Tue 15:00Wed 12:00Wed 13:00Wed 15:00Wed 16:00Thu 09:00Thu 12:00Thu 13:00Thu 15:00Thu 16:00Fri 09:00Fri 12:00Fri 13:00Fri 15:00Fri 16:00Sat 09:00Sat 12:00Sat 13:00Sat 15:00Sat 16:00
Room 09,👥 23,Mon 15:00Mon 16:00Tue 09:00Tue 13:00Tue 15:00Wed 13:00Wed 15:00Thu 13:00Thu 16:00Fri 09:00Fri 13:00Fri 15:00Fri 16:00Sat 13:00Sat 15:00
Room 10,👥 34,Mon 15:00Mon 16:00Tue 12:00Tue 13:00Tue 15:00Wed 12:00Wed 13:00Wed 15:00Wed 16:00Thu 09:00Thu 13:00Thu 16:00Fri 09:00Fri 12:00Fri 16:00Sat 12:00Sat 13:00
Room 11,👥 30,Fri 12:00Fri 13:00Fri 15:00Fri 16:00
Room 12,👥 36,Mon 09:00Mon 12:00Mon 13:00Mon 15:00Mon 16:00Thu 09:00Thu 12:00Thu 13:00Thu 15:00Thu 16:00


### 8e. Section 4 — Schedule verification result

In [21]:
css = '''
<style>
  .tt-verify{margin-top:16px;border-radius:10px;overflow:hidden}
  .tt-verify-ok{border:1px solid #bbf7d0}
  .tt-verify-fail{border:1px solid #fca5a5}
  .tt-verify-hok{background:#dcfce7;padding:10px 16px;font-size:13px;font-weight:600;color:#14532d;border-bottom:1px solid #bbf7d0}
  .tt-verify-hfail{background:#fee2e2;padding:10px 16px;font-size:13px;font-weight:600;color:#7f1d1d;border-bottom:1px solid #fca5a5}
  .tt-verify-body{padding:10px 16px;background:#fff;font-size:12px}
  .tt-checks{display:flex;flex-wrap:wrap;gap:8px}
  .tt-check{display:flex;align-items:center;gap:5px;font-size:11px;color:#374151;background:#f9fafb;padding:4px 10px;border-radius:20px;border:1px solid #e5e7eb}
  .tt-vrow{padding:6px 0;border-bottom:1px solid #fee2e2;color:#991b1b}
  .tt-vrow:last-child{border-bottom:none}
</style>
'''

display(HTML(css))

# Section 4: Schedule verification
html = '<div class="tt-wrap">'
checks = [
    'No room double-booking',
    'No professor time conflict',
    'Room capacity respected',
    'Preferred days honoured',
    'Preferred times honoured',
    'Room availability respected',
]
if not violations:
    html += ('<div class="tt-verify tt-verify-ok">'
             '<div class="tt-verify-hok">&#10003; Schedule Verified - No Violations Found</div>'
             '<div class="tt-verify-body"><div class="tt-checks">')
    for chk in checks:
        html += f'<div class="tt-check"><span style="color:#16a34a">&#10003;</span> {chk}</div>'
    html += '</div></div></div>'
else:
    html += ('<div class="tt-verify tt-verify-fail">'
             f'<div class="tt-verify-hfail">&#10007; {len(violations)} '
             f'Violation{"s" if len(violations)!=1 else ""} Detected</div>'
             '<div class="tt-verify-body">')
    for v in violations:
        html += f'<div class="tt-vrow">&#8227; {v}</div>'
    html += '</div></div>'

html += '</div>'
display(HTML(html))